# Find and manage your Runs

Every `sf.evaluate(...)` creates an Engine-owned Run: an id, a state, and an outcome and cost for
each Candidate. This notebook starts one, finds it again from its id, follows its progress, opens
its Report, and cancels a Run.

Listing and inspecting Runs make no model calls. The evaluation is **off by default**, and
cancellation needs an explicit Run id.

## Before running

The next cell picks where this notebook runs: `TARGET = "local"` (the default)
uses your `screamingface up` stack, and `TARGET = "hosted"` uses
`fusion.screamingface.ai`.

**Local.** Install the local services with
`pip install "screamingface[runtime,notebook]"`, then from a terminal:

```bash
screamingface prepare ifeval  # first run only: download the pinned Benchmark assets
screamingface up
screamingface status
```

Use `screamingface logs` to inspect startup failures and `screamingface down` when
finished. Stack management stays outside the notebook so **Run All** never starts or
stops local services. Working from a checkout? `just local-stack-notebooks` in
`packages/screamingface/` does every local step — assets, stack, and Jupyter — in one
command.

**Hosted.** Nothing to start: `pip install "screamingface[notebook]"` is enough, and
`sf.connect()` opens the login in your browser.

In [ ]:
import screamingface as sf

TARGET = "local"  # "local": your `screamingface up` stack · "hosted": fusion.screamingface.ai

if TARGET == "local":
    sf.configure(engine_url="http://127.0.0.1:9108", scoreboard_url="http://127.0.0.1:9106")
elif TARGET == "hosted":
    sf.configure(
        engine_url="https://fusion.screamingface.ai",
        scoreboard_url="https://leaderboard.screamingface.ai",
    )
else:
    raise ValueError(f'TARGET must be "local" or "hosted", not {TARGET!r}')

Local mode uses the local Engine's Run database. `TARGET = "hosted"` uses
`fusion.screamingface.ai` and needs your Cloudflare Access login. Run metadata is visible on any
machine signed in as the same owner, but `Run.report()` opens only a Report saved on this
machine.

`sf.connect()` shows the provider panel: on hosted it first asks you to log in, then on either
target connect the provider of the Model in section 1 (OpenRouter) before evaluating.

In [ ]:
sf.connect()

## 1. Start an evaluation (paid, off by default)

Set `RUN_EVALUATION = True` only when you are ready to make model calls. `report.run_id` is the id
of the evaluation's Run. Each Candidate also has its own `attempt_id`: one attempt of one
Candidate, not the Run (`CandidateResult.run_id` is its deprecated old name). An Engine without
Runs still evaluates, but `report.run_id` is `None`.

In [ ]:
MODEL = sf.Model("openrouter/google/gemini-3-flash-preview")
RUN_EVALUATION = False

report = sf.evaluate(MODEL, benchmark="ifeval", limit=1) if RUN_EVALUATION else None
if report is None:
    started = "No paid evaluation started. Set RUN_EVALUATION = True to opt in."
else:
    started = {
        "run_id": report.run_id,
        "attempts": [candidate.attempt_id for candidate in report.candidates],
    }
started

## 2. Find your Runs

`sf.runs.list()` returns your Runs newest first. Filter by `state` to find active work, or by
`since` (a timezone-aware datetime) for recent Runs. Listing works without starting anything.

In [ ]:
runs = sf.runs.list(limit=10)
[
    {
        "id": item.id,
        "benchmark": item.benchmark,
        "state": item.state,
        "created_at": item.created_at,
        "finished_at": item.finished_at,
        "cost_usd": item.cost_usd,
    }
    for item in runs
]

In [ ]:
from datetime import UTC, datetime, timedelta

{
    "running": [item.id for item in sf.runs.list(state="running", limit=10)],
    "last_24h": [item.id for item in sf.runs.list(since=datetime.now(UTC) - timedelta(days=1))],
}

## 3. Reopen a Run and follow its progress

`sf.runs.get(run_id)` returns a snapshot that never changes. `run.refresh()` reads the Engine
again, and `run.progress()` counts finished Candidates and known spend without waiting. A
Candidate that did not succeed carries an `error_code`. This uses the Run started above, or else
your newest Run.

In [ ]:
if report is not None and report.run_id is not None:
    run_id = report.run_id
else:
    run_id = runs[0].id if runs else None
run = sf.runs.get(run_id) if run_id is not None else None

if run is None:
    snapshot = "No Runs yet. Opt in to the evaluation in section 1 to create one."
else:
    run = run.refresh()
    progress = run.progress()
    snapshot = {
        "id": run.id,
        "state": run.state,
        "finished_at": run.finished_at,
        "progress": f"{progress.done}/{progress.total} Candidates, ${progress.cost_usd}",
        "candidates": [
            {
                "name": item.name,
                "state": item.state,
                "cost_usd": item.cost_usd,
                "error_code": item.error_code,
            }
            for item in run.candidates
        ],
    }
snapshot

`run.wait()` polls until the Run finishes and returns it; it raises `TimeoutError` past
`timeout`. It blocks the cell, so it is off by default.

In [ ]:
WAIT_FOR_COMPLETION = False
finished_run = run.wait(timeout=300) if WAIT_FOR_COMPLETION and run is not None else None
finished_run.state if finished_run is not None else "Not waiting."

## 4. Open the Run's Report

`Run.report()` reads the results saved on this machine; the Engine does not keep model output.
It raises `ExecutionError` with `code="run_not_finished"` while the Run is running, and
`code="results_not_local"` when another machine ran it.

In [ ]:
if run is None:
    recovered_report = "No Run to open yet."
else:
    reopened_run = sf.runs.get(run.id)
    try:
        recovered_report = reopened_run.report()
    except sf.ExecutionError as exc:
        recovered_report = f"{exc.code}: {exc.hint}"
recovered_report

## 5. Cancel a Run (explicit id required)

Cancellation is run-wide: it stops every Candidate still running and refuses the Run's later
starts (`run_canceled`). A finished Run is left as it is. Pick an id from section 2 only when you
intend to stop that Run.

In [ ]:
CANCEL_RUN_ID = None

if CANCEL_RUN_ID is not None:
    run_to_cancel = sf.runs.get(CANCEL_RUN_ID)
    run_to_cancel.cancel()
    cancellation = f"Cancellation requested for {CANCEL_RUN_ID}"
else:
    cancellation = "No Run cancelled. Set CANCEL_RUN_ID to the Run you intend to stop."
cancellation

Once its Candidates have stopped, a cancelled Run reads `"canceled"`.

In [ ]:
sf.runs.get(CANCEL_RUN_ID).state if CANCEL_RUN_ID is not None else "Nothing cancelled."

## 6. When a Run cannot be read

`sf.runs.get` raises `KeyError` for an unknown id, or another owner's. Against an Engine that does
not serve Runs, `sf.runs.list` and `sf.runs.get` raise `ExecutionError` with
`code="runs_unavailable"`. In async code, use `await sf.runs.list_async(...)` and
`await sf.runs.get_async(run_id)`.

In [ ]:
unknown = "found"
try:
    sf.runs.get("run_does_not_exist")
except KeyError:
    unknown = "KeyError: no such Run for this owner"
unknown